# Flipkart Traffic Demand — Optuna Hyperparameter Tuning
Starting from the 87.52-scoring baseline. Optuna searches per-road-type hyperparameters
using Day 48 → Day 49 cross-validation, then retrains final models on 100% of data
with the best params found.

## 1. Imports & Config

In [1]:
import warnings
import numpy as np
import pandas as pd
from sklearn.preprocessing import PowerTransformer
from sklearn.metrics import r2_score
import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.4f}".format)

SEED = 42
np.random.seed(SEED)

def competition_score(actual, predicted):
    return max(0, 100 * r2_score(actual, predicted))

/home/arnab/miniconda3/envs/flipkark/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load Data

In [2]:
df_raw = pd.read_csv("raw.csv")
df = df_raw.copy()
print(f"Shape: {df.shape}")

Shape: (77299, 11)


## 3. Data Cleaning

In [3]:
geo_roadtype_mode = (
    df.groupby("geohash")["RoadType"]
    .agg(lambda x: x.mode()[0] if x.notna().any() else np.nan)
)
df["RoadType"] = df["RoadType"].fillna(df["geohash"].map(geo_roadtype_mode))
df["RoadType"] = df["RoadType"].fillna(df["RoadType"].mode()[0])

df["Weather"] = df["Weather"].fillna("Unknown")

df["_hour_tmp"] = df["timestamp"].str.split(":").str[0].astype(int)
geo_hour_temp_median = df.groupby(["geohash", "_hour_tmp"])["Temperature"].median()

def fill_temperature(row):
    if pd.isnull(row["Temperature"]):
        return geo_hour_temp_median.get((row["geohash"], row["_hour_tmp"]), np.nan)
    return row["Temperature"]

df["Temperature"] = df.apply(fill_temperature, axis=1)
df["Temperature"] = df["Temperature"].fillna(df["Temperature"].median())
df.drop(columns=["_hour_tmp"], inplace=True)

assert df.isnull().sum().sum() == 0
print("✓ No nulls remaining.")

✓ No nulls remaining.


## 4. Feature Engineering

In [4]:
df["hour"]         = df["timestamp"].str.split(":").str[0].astype(int)
df["minute"]       = df["timestamp"].str.split(":").str[1].astype(int)
df["time_minutes"] = df["hour"] * 60 + df["minute"]

PERIOD = 1440
df["time_sin"] = np.sin(2 * np.pi * df["time_minutes"] / PERIOD)
df["time_cos"] = np.cos(2 * np.pi * df["time_minutes"] / PERIOD)

df = df.sort_values(["geohash", "day", "time_minutes"]).reset_index(drop=True)
df["geo5"] = df["geohash"].str[:5]

## 5. Build Train / Val Splits for Optuna
Optuna needs a proper holdout to evaluate each trial without overfitting the search.
We use the natural Day 48 → Day 49 split for this — same as the original validation setup.
All encodings are computed from Day 48 only and applied to Day 49, preventing leakage.

In [5]:
df_train = df[df["day"] == 48].copy()
df_val   = df[df["day"] == 49].copy()

# ── Target transform — fit on Day 48 only ────────────────────────────────────
pt_yeo_cv = PowerTransformer(method="yeo-johnson", standardize=False)
df_train["yeo_demand"] = pt_yeo_cv.fit_transform(df_train[["demand"]]).flatten()
df_val["yeo_demand"]   = pt_yeo_cv.transform(df_val[["demand"]]).flatten()

# ── Geohash target encoding — from Day 48 only ───────────────────────────────
global_mean_cv = df_train["demand"].mean()
stats_cv = df_train.groupby("geohash")["demand"].agg(["mean", "count"])
stats_cv["encoded"] = (
    (stats_cv["count"] * stats_cv["mean"] + 15 * global_mean_cv)
    / (stats_cv["count"] + 15)
)
geo_mean_map_cv = stats_cv["encoded"].to_dict()
df_train["geohash_encoded"] = df_train["geohash"].map(geo_mean_map_cv).fillna(global_mean_cv)
df_val["geohash_encoded"]   = df_val["geohash"].map(geo_mean_map_cv).fillna(global_mean_cv)

# ── Spatial neighbour — from Day 48 only ─────────────────────────────────────
geo5_hour_cv = (
    df_train.groupby(["geo5", "hour"])["demand"]
    .mean().rename("geo5_hour_demand_mean").reset_index()
)
df_train = df_train.merge(geo5_hour_cv, on=["geo5", "hour"], how="left")
df_val   = df_val.merge(geo5_hour_cv, on=["geo5", "hour"], how="left")
df_train["geo5_hour_demand_mean"] = df_train["geo5_hour_demand_mean"].fillna(global_mean_cv)
df_val["geo5_hour_demand_mean"]   = df_val["geo5_hour_demand_mean"].fillna(global_mean_cv)

# ── Categoricals ─────────────────────────────────────────────────────────────
df_train["LargeVehicles_enc"] = (df_train["LargeVehicles"] == "Allowed").astype(int)
df_val["LargeVehicles_enc"]   = (df_val["LargeVehicles"]   == "Allowed").astype(int)
df_train["Landmarks_enc"] = (df_train["Landmarks"] == "Yes").astype(int)
df_val["Landmarks_enc"]   = (df_val["Landmarks"]   == "Yes").astype(int)

df_train = pd.get_dummies(df_train, columns=["RoadType"], drop_first=True)
df_val   = pd.get_dummies(df_val,   columns=["RoadType"], drop_first=True)
for col in df_train.columns:
    if col not in df_val.columns:
        df_val[col] = 0

df_train = pd.get_dummies(df_train, columns=["Weather"], drop_first=True)
df_val   = pd.get_dummies(df_val,   columns=["Weather"], drop_first=True)
for col in df_train.columns:
    if col not in df_val.columns:
        df_val[col] = 0

# ── Native geohash category ───────────────────────────────────────────────────
df_train["geohash_cat"] = df_train["geohash"].astype("category")
known_cats_cv = df_train["geohash_cat"].cat.categories
df_val["geohash_cat"] = pd.Categorical(df_val["geohash"], categories=known_cats_cv)

# ── Feature list ──────────────────────────────────────────────────────────────
FEATURES_CV = [
    "time_sin", "time_cos", "time_minutes", "hour",
    "NumberofLanes", "LargeVehicles_enc", "Landmarks_enc",
    *[c for c in df_train.columns if c.startswith("RoadType_")],
    *[c for c in df_train.columns if c.startswith("Weather_")],
    "Temperature",
    "geo5_hour_demand_mean", "geohash_cat",
]

print(f"Train: {df_train.shape}, Val: {df_val.shape}")
print(f"CV features: {len(FEATURES_CV)}")

Train: (69427, 27), Val: (7872, 27)
CV features: 16


## 6. Road Type Masks for CV

In [6]:
def get_road_masks(df_):
    res = df_.get("RoadType_Residential", pd.Series(0, index=df_.index)).astype(bool)
    st  = df_.get("RoadType_Street",      pd.Series(0, index=df_.index)).astype(bool)
    hwy = ~(res | st)
    return {"Highway": hwy, "Street": st, "Residential": res}

train_masks = get_road_masks(df_train)
val_masks   = get_road_masks(df_val)

for rt, mask in train_masks.items():
    print(f"  {rt}: train={mask.sum():,}  val={val_masks[rt].sum():,}")

  Highway: train=3,140  val=453
  Street: train=3,414  val=503
  Residential: train=62,873  val=6,916


## 7. Optuna Objective
One study per road type. Each trial trains LightGBM on Day 48 and evaluates on Day 49.
The objective maximises the competition score (100 × R²).

**Search space rationale:**
- `num_leaves` 31–255: controls model complexity; Highway needs higher than Residential
- `min_data_in_leaf` 10–100: key guard against single-geohash memorisation
- `learning_rate` 0.01–0.15: with fixed 500 rounds, lower LR = more regularised
- `reg_alpha`/`reg_lambda` 1e-8–10: wide range — Phase 7 showed too much L2 kills peaks
- `feature_fraction`/`bagging_fraction` 0.5–1.0: controls variance

In [7]:
def make_objective(road_type, df_train, df_val, train_masks, val_masks,
                   FEATURES_CV, pt_yeo_cv):
    def objective(trial):
        params = {
            "objective":        "regression",
            "metric":           "rmse",
            "verbose":          -1,
            "seed":             SEED,
            "bagging_seed":     SEED,
            "feature_seed":     SEED,
            "num_leaves":       trial.suggest_int("num_leaves", 31, 255),
            "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
            "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 10, 100),
            "feature_fraction": trial.suggest_float("feature_fraction", 0.5, 1.0),
            "bagging_fraction": trial.suggest_float("bagging_fraction", 0.5, 1.0),
            "bagging_freq":     5,
            "reg_alpha":        trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            "reg_lambda":       trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        }

        tr_mask = train_masks[road_type]
        vl_mask = val_masks[road_type]

        sub_tr = df_train[tr_mask]
        sub_vl = df_val[vl_mask]

        lgb_tr = lgb.Dataset(
            sub_tr[FEATURES_CV], label=sub_tr["yeo_demand"],
            categorical_feature=["geohash_cat"], free_raw_data=False,
        )
        lgb_vl = lgb.Dataset(
            sub_vl[FEATURES_CV], label=sub_vl["yeo_demand"],
            categorical_feature=["geohash_cat"],
            reference=lgb_tr, free_raw_data=False,
        )

        model = lgb.train(
            params,
            train_set       = lgb_tr,
            num_boost_round = 500,
            valid_sets      = [lgb_vl],
            callbacks       = [lgb.early_stopping(50, verbose=False),
                               lgb.log_evaluation(-1)],
        )

        raw_preds = pt_yeo_cv.inverse_transform(
            model.predict(sub_vl[FEATURES_CV]).reshape(-1, 1)
        ).flatten()
        preds = np.clip(raw_preds, 0, 1)
        score = competition_score(sub_vl["demand"].values, preds)
        return score  # maximise

    return objective

## 8. Run Optuna — One Study Per Road Type
`n_trials=50` gives a thorough search in reasonable time (~5–10 min).
Increase to 100 if you have more time.

In [8]:
N_TRIALS = 200
best_params_per_road = {}

for road_type in ["Highway", "Street", "Residential"]:
    print(f"\n{'='*50}")
    print(f"  Tuning: {road_type}  ({N_TRIALS} trials)")
    print(f"{'='*50}")

    study = optuna.create_study(
        direction  = "maximize",
        sampler    = optuna.samplers.TPESampler(seed=SEED),
    )
    study.optimize(
        make_objective(road_type, df_train, df_val, train_masks, val_masks,
                       FEATURES_CV, pt_yeo_cv),
        n_trials   = N_TRIALS,
        show_progress_bar = True,
    )

    best_params_per_road[road_type] = study.best_params
    print(f"\n  Best score: {study.best_value:.4f}")
    print(f"  Best params:")
    for k, v in study.best_params.items():
        print(f"    {k}: {v}")


  Tuning: Highway  (200 trials)


Best trial: 0. Best value: 0: 100%|██████████| 200/200 [01:00<00:00,  3.33it/s]



  Best score: 0.0000
  Best params:
    num_leaves: 115
    learning_rate: 0.13125830316209655
    min_data_in_leaf: 76
    feature_fraction: 0.7993292420985183
    bagging_fraction: 0.5780093202212182
    reg_alpha: 2.5348407664333426e-07
    reg_lambda: 3.3323645788192616e-08

  Tuning: Street  (200 trials)


Best trial: 0. Best value: 0: 100%|██████████| 200/200 [01:11<00:00,  2.81it/s]



  Best score: 0.0000
  Best params:
    num_leaves: 115
    learning_rate: 0.13125830316209655
    min_data_in_leaf: 76
    feature_fraction: 0.7993292420985183
    bagging_fraction: 0.5780093202212182
    reg_alpha: 2.5348407664333426e-07
    reg_lambda: 3.3323645788192616e-08

  Tuning: Residential  (200 trials)


Best trial: 173. Best value: 36.1661: 100%|██████████| 200/200 [06:10<00:00,  1.85s/it]


  Best score: 36.1661
  Best params:
    num_leaves: 31
    learning_rate: 0.14946323677386525
    min_data_in_leaf: 98
    feature_fraction: 0.6124241022788327
    bagging_fraction: 0.9698989584823197
    reg_alpha: 2.057050444861167e-07
    reg_lambda: 9.437989711711729e-07


## 9. Tuning Summary

In [9]:
print("\n=== BEST PARAMS PER ROAD TYPE ===\n")
for road_type, params in best_params_per_road.items():
    print(f"{road_type}:")
    for k, v in params.items():
        v_fmt = f"{v:.6f}" if isinstance(v, float) else str(v)
        print(f"  {k}: {v_fmt}")
    print()


=== BEST PARAMS PER ROAD TYPE ===

Highway:
  num_leaves: 115
  learning_rate: 0.131258
  min_data_in_leaf: 76
  feature_fraction: 0.799329
  bagging_fraction: 0.578009
  reg_alpha: 0.000000
  reg_lambda: 0.000000

Street:
  num_leaves: 115
  learning_rate: 0.131258
  min_data_in_leaf: 76
  feature_fraction: 0.799329
  bagging_fraction: 0.578009
  reg_alpha: 0.000000
  reg_lambda: 0.000000

Residential:
  num_leaves: 31
  learning_rate: 0.149463
  min_data_in_leaf: 98
  feature_fraction: 0.612424
  bagging_fraction: 0.969899
  reg_alpha: 0.000000
  reg_lambda: 0.000001



## 10. Full-Dataset Prep
Refit all encodings on 100% of data (Day 48 + Day 49) for final model training.

In [10]:
df_full = df.copy()

global_demand_mean = df_full["demand"].mean()
stats = df_full.groupby("geohash")["demand"].agg(["mean", "count"])
stats["encoded"] = (
    (stats["count"] * stats["mean"] + 15 * global_demand_mean)
    / (stats["count"] + 15)
)
geo_mean_map = stats["encoded"].to_dict()
df_full["geohash_encoded"] = df_full["geohash"].map(geo_mean_map).fillna(global_demand_mean)

geo5_hour_mean = (
    df_full.groupby(["geo5", "hour"])["demand"]
    .mean().rename("geo5_hour_demand_mean").reset_index()
)
df_full = df_full.merge(geo5_hour_mean, on=["geo5", "hour"], how="left")
df_full["geo5_hour_demand_mean"] = df_full["geo5_hour_demand_mean"].fillna(global_demand_mean)

df_full["LargeVehicles_enc"] = (df_full["LargeVehicles"] == "Allowed").astype(int)
df_full["Landmarks_enc"]     = (df_full["Landmarks"] == "Yes").astype(int)
df_full = pd.get_dummies(df_full, columns=["RoadType"], drop_first=True)
df_full = pd.get_dummies(df_full, columns=["Weather"],  drop_first=True)

pt_yeo = PowerTransformer(method="yeo-johnson", standardize=False)
df_full["yeo_demand"] = pt_yeo.fit_transform(df_full[["demand"]]).flatten()
yeo_min = df_full["yeo_demand"].min()
yeo_max = df_full["yeo_demand"].max()

df_full["geohash_cat"] = df_full["geohash"].astype("category")
known_cats = df_full["geohash_cat"].cat.categories

FEATURES_LGB = [
    "time_sin", "time_cos", "time_minutes", "hour",
    "NumberofLanes", "LargeVehicles_enc", "Landmarks_enc",
    *[c for c in df_full.columns if c.startswith("RoadType_")],
    *[c for c in df_full.columns if c.startswith("Weather_")],
    "Temperature",
    "geo5_hour_demand_mean", "geohash_cat",
]

missing = [f for f in FEATURES_LGB if f not in df_full.columns]
assert len(missing) == 0, f"Missing: {missing}"
print(f"df_full shape: {df_full.shape}")
print(f"Features: {len(FEATURES_LGB)}")

df_full shape: (77299, 27)
Features: 16


## 11. Train Final Models with Best Params
Uses the tuned hyperparameters, trains on 100% of data.
`num_boost_round=500` — same ceiling used during Optuna search, so early stopping
behaviour is consistent with what was evaluated.

In [11]:
models_per_road = {}

for road_type in ["Highway", "Street", "Residential"]:
    print(f"\nTraining final model: {road_type}")

    res_mask = df_full.get("RoadType_Residential", pd.Series(0, index=df_full.index)).astype(bool)
    str_mask = df_full.get("RoadType_Street",      pd.Series(0, index=df_full.index)).astype(bool)

    if road_type == "Residential":
        mask = res_mask
    elif road_type == "Street":
        mask = str_mask
    else:
        mask = ~(res_mask | str_mask)

    sub = df_full[mask]
    print(f"  Rows: {len(sub):,}")

    # Build final params: best from Optuna + fixed keys
    final_params = {
        "objective":    "regression",
        "metric":       "rmse",
        "verbose":      -1,
        "seed":         SEED,
        "bagging_seed": SEED,
        "feature_seed": SEED,
        "bagging_freq": 5,
        **best_params_per_road[road_type],
    }

    lgb_data = lgb.Dataset(
        sub[FEATURES_LGB], label=sub["yeo_demand"],
        categorical_feature=["geohash_cat"], free_raw_data=False,
    )
    model = lgb.train(
        params          = final_params,
        train_set       = lgb_data,
        num_boost_round = 500,
    )
    models_per_road[road_type] = model
    print(f"  ✓ Done")

print("\nAll final models trained.")


Training final model: Highway
  Rows: 3,593
  ✓ Done

Training final model: Street
  Rows: 3,917
  ✓ Done

Training final model: Residential
  Rows: 69,789
  ✓ Done

All final models trained.


## 12. Indicative Score on Day 49
(In-sample — model trained on Day 49. Confirms predictions are sensible.)

In [12]:
df_val_check = df_full[df_full["day"] == 49].copy()
y_val_raw    = df_val_check["demand"].values
y_pred_check = np.zeros(len(df_val_check))

res_v = df_val_check.get("RoadType_Residential", pd.Series(0, index=df_val_check.index)).astype(bool)
str_v = df_val_check.get("RoadType_Street",      pd.Series(0, index=df_val_check.index)).astype(bool)
hwy_v = ~(res_v | str_v)

for road_type, mask in [("Highway", hwy_v), ("Street", str_v), ("Residential", res_v)]:
    if road_type not in models_per_road:
        continue
    idx = df_val_check.index[mask]
    if len(idx) == 0:
        continue
    yeo_preds = np.clip(models_per_road[road_type].predict(df_val_check.loc[idx, FEATURES_LGB]),
                        yeo_min, yeo_max)
    y_pred_check[mask.values] = np.clip(
        pt_yeo.inverse_transform(yeo_preds.reshape(-1, 1)).flatten(), 0, 1
    )

print(f"Day 49 indicative score: {competition_score(y_val_raw, y_pred_check):.4f}")

Day 49 indicative score: 91.7663


## 13. Test Pipeline

In [13]:
def align_columns(df_enc, train_columns):
    for col in train_columns:
        if col not in df_enc.columns:
            print(f"  [WARNING] '{col}' missing — adding zeros.")
            df_enc[col] = 0
    extra = set(df_enc.columns) - set(train_columns)
    if extra:
        df_enc.drop(columns=list(extra), inplace=True)
    return df_enc[train_columns]


def run_test_pipeline(
    test_path, df_full, geo_mean_map, geo5_hour_mean,
    models_per_road, pt_yeo, FEATURES_LGB, known_cats,
    global_demand_mean, yeo_min, yeo_max,
):
    print("Step 1: Loading...")
    df_test = pd.read_csv(test_path)
    print(f"  Shape: {df_test.shape}")

    print("\nStep 2: Cleaning...")
    res_tr = df_full.get("RoadType_Residential", pd.Series(0, index=df_full.index)).astype(bool)
    str_tr = df_full.get("RoadType_Street",      pd.Series(0, index=df_full.index)).astype(bool)
    temp_rt = pd.Series("Highway", index=df_full.index)
    temp_rt.loc[res_tr] = "Residential"
    temp_rt.loc[str_tr] = "Street"
    geo_rt_mode = temp_rt.groupby(df_full["geohash"]).agg(lambda x: x.mode()[0])
    df_test["RoadType"] = (
        df_test["RoadType"].fillna(df_test["geohash"].map(geo_rt_mode)).fillna("Highway")
    )
    df_test["Weather"] = df_test["Weather"].fillna("Unknown")

    df_test["_hour_tmp"] = df_test["timestamp"].str.split(":").str[0].astype(int)
    geo_hr_temp = df_full.groupby(["geohash", "hour"])["Temperature"].median()
    df_test["Temperature"] = df_test.apply(
        lambda r: geo_hr_temp.get((r["geohash"], r["_hour_tmp"]), np.nan)
        if pd.isnull(r.get("Temperature", np.nan)) else r.get("Temperature", np.nan),
        axis=1
    ).fillna(df_full["Temperature"].median())
    df_test.drop(columns=["_hour_tmp"], inplace=True)

    print("\nStep 3: Features...")
    df_test["hour"]         = df_test["timestamp"].str.split(":").str[0].astype(int)
    df_test["minute"]       = df_test["timestamp"].str.split(":").str[1].astype(int)
    df_test["time_minutes"] = df_test["hour"] * 60 + df_test["minute"]
    PERIOD = 1440
    df_test["time_sin"] = np.sin(2 * np.pi * df_test["time_minutes"] / PERIOD)
    df_test["time_cos"] = np.cos(2 * np.pi * df_test["time_minutes"] / PERIOD)

    df_test["geohash_encoded"] = df_test["geohash"].map(geo_mean_map).fillna(global_demand_mean)
    df_test["geo5"] = df_test["geohash"].str[:5]
    df_test = df_test.merge(geo5_hour_mean, on=["geo5", "hour"], how="left")
    df_test["geo5_hour_demand_mean"] = df_test["geo5_hour_demand_mean"].fillna(global_demand_mean)

    df_test["LargeVehicles_enc"] = (df_test["LargeVehicles"] == "Allowed").astype(int)
    df_test["Landmarks_enc"]     = (df_test["Landmarks"] == "Yes").astype(int)
    df_test = pd.get_dummies(df_test, columns=["RoadType"], drop_first=True)
    df_test = pd.get_dummies(df_test, columns=["Weather"],  drop_first=True)

    feat_no_cat = [f for f in FEATURES_LGB if f != "geohash_cat"]
    X_test = align_columns(df_test.copy(), feat_no_cat)
    X_test["geohash_cat"] = pd.Categorical(df_test["geohash"], categories=known_cats)

    print("\nStep 4: Predicting...")
    y_pred = np.zeros(len(df_test))

    res_mask = df_test.get("RoadType_Residential", pd.Series(0, index=df_test.index)).astype(bool)
    str_mask = df_test.get("RoadType_Street",      pd.Series(0, index=df_test.index)).astype(bool)
    hwy_mask = ~(res_mask | str_mask)

    for road_type, mask in [("Highway", hwy_mask), ("Street", str_mask), ("Residential", res_mask)]:
        if road_type not in models_per_road:
            continue
        idx = df_test.index[mask]
        if len(idx) == 0:
            continue
        yeo_preds = np.clip(models_per_road[road_type].predict(X_test.loc[idx]), yeo_min, yeo_max)
        y_pred[mask.values] = np.clip(
            pt_yeo.inverse_transform(yeo_preds.reshape(-1, 1)).flatten(), 0, 1
        )
        print(f"  {road_type}: {mask.sum()} rows")

    print("\nStep 5: Saving...")
    submission = pd.DataFrame({"Index": df_test["Index"], "demand": y_pred})
    assert submission["demand"].isna().sum() == 0, "NaN predictions!"
    assert (submission["demand"] >= 0).all(),      "Negative predictions!"
    assert (submission["demand"] <= 1).all(),      "Predictions above 1!"
    assert len(submission) == len(df_test),        "Row count mismatch!"

    submission.to_csv("submission.csv", index=False)
    print(f"  ✓ submission.csv saved  ({len(submission):,} rows)")
    print(submission["demand"].describe().round(4))
    return submission

## 14. Generate Submission

In [14]:
submission = run_test_pipeline(
    test_path          = "test.csv",
    df_full            = df_full,
    geo_mean_map       = geo_mean_map,
    geo5_hour_mean     = geo5_hour_mean,
    models_per_road    = models_per_road,
    pt_yeo             = pt_yeo,
    FEATURES_LGB       = FEATURES_LGB,
    known_cats         = known_cats,
    global_demand_mean = global_demand_mean,
    yeo_min            = yeo_min,
    yeo_max            = yeo_max,
)

Step 1: Loading...
  Shape: (41778, 10)

Step 2: Cleaning...

Step 3: Features...

Step 4: Predicting...
  Highway: 4285 rows
  Street: 3411 rows
  Residential: 34082 rows

Step 5: Saving...
  ✓ submission.csv saved  (41,778 rows)
count   41778.0000
mean        0.1159
std         0.1597
min         0.0000
25%         0.0241
50%         0.0543
75%         0.1193
max         1.0000
Name: demand, dtype: float64


## 15. Submission Audit

In [15]:
sub = pd.read_csv("submission.csv")
print(f"  Rows:        {len(sub):,}")
print(f"  Columns:     {sub.columns.tolist()}")
print(f"  NaNs:        {sub.isnull().sum().values}")
print(f"  Min demand:  {sub['demand'].min():.6f}")
print(f"  Max demand:  {sub['demand'].max():.6f}")
print()
print(sub.head(10))

  Rows:        41,778
  Columns:     ['Index', 'demand']
  NaNs:        [0 0]
  Min demand:  0.000001
  Max demand:  1.000000

   Index  demand
0      0  0.0400
1      1  0.0191
2      2  0.0069
3      3  0.0282
4      4  0.0499
5      5  0.0322
6      6  0.0308
7      7  0.0932
8      8  0.0335
9      9  0.0686
